# 📊 統計学の基礎：母集団・標本・観測値

このノートブックでは、統計学における3つの重要な概念を視覚的に学びます。

| 概念 | 定義 | 例 |
|------|------|----|
| **母集団 (Population)** | 調査・分析の対象となるすべての個体の集合 | J国全体の成人男性の身長 |
| **標本 (Sample)** | 母集団から抽出した一部の個体の集合 | J国のある地域で調査した500人の身長 |
| **観測値 (Observation)** | 標本の中の各個体から実際に測定した具体的な値 | 175cm, 168cm, 182cm, ... |

---

In [ ]:
# ライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import pandas as pd
from scipy import stats

# 日本語フォント設定（Colab用）
!pip install japanize-matplotlib -q
import japanize_matplotlib

plt.rcParams['figure.dpi'] = 120
np.random.seed(42)
print("✅ 準備完了！")

## 🏘️ Step 1: 母集団を作る

「J国の成人男性の身長」を母集団として設定します。

- **母集団の平均 (μ)**: 171 cm
- **母集団の標準偏差 (σ)**: 6 cm
- **母集団のサイズ (N)**: 10,000人

In [ ]:
# ===== 母集団の設定 =====
mu = 171        # 母平均 (cm)
sigma = 6       # 母標準偏差 (cm)
N = 10000       # 母集団サイズ

# 母集団データを生成（正規分布）
population = np.random.normal(mu, sigma, N)

print("=" * 40)
print("📦 母集団 (Population) の情報")
print("=" * 40)
print(f"  サイズ (N)       : {N:,} 人")
print(f"  真の平均 (μ)     : {mu} cm")
print(f"  真の標準偏差 (σ) : {sigma} cm")
print(f"  実際の平均       : {population.mean():.2f} cm")
print(f"  実際の標準偏差   : {population.std():.2f} cm")
print()
print("最初の10人の身長（観測値の例）:")
print([f"{x:.1f}" for x in population[:10]])

In [ ]:
# 母集団の分布を可視化
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(population, bins=60, color='#4A90D9', alpha=0.7, edgecolor='white', linewidth=0.5)
ax.axvline(population.mean(), color='#E74C3C', linewidth=2.5, linestyle='--', label=f'母平均 μ = {mu} cm')

ax.set_title('【母集団】J国の成人男性の身長分布 (N=10,000人)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('身長 (cm)', fontsize=12)
ax.set_ylabel('人数', fontsize=12)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# 注釈
ax.text(0.97, 0.92, f'N = {N:,}人\nμ = {mu}cm\nσ = {sigma}cm',
        transform=ax.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='#E74C3C', alpha=0.9),
        fontsize=10)

plt.tight_layout()
plt.show()
print("\n👉 これが母集団全体の分布です。現実では全員を調査することは不可能なので、「標本」を使います。")

## 🎯 Step 2: 母集団から標本を抽出する

現実には母集団全員を調査することは困難です。そこで母集団から一部を**無作為抽出（ランダムサンプリング）**します。

抽出した各データが「**観測値**」、集まり全体が「**標本**」です。

In [ ]:
# ===== 標本の抽出 =====
n = 50   # 標本サイズ

# 無作為抽出（ランダムサンプリング）
sample_indices = np.random.choice(N, n, replace=False)
sample = population[sample_indices]

print("=" * 40)
print("🎯 標本 (Sample) の情報")
print("=" * 40)
print(f"  標本サイズ (n)   : {n} 人")
print(f"  標本平均 (x̄)     : {sample.mean():.2f} cm  ← 母平均 μ={mu}cm の推定値")
print(f"  標本標準偏差 (s) : {sample.std(ddof=1):.2f} cm  ← 母標準偏差 σ={sigma}cm の推定値")
print()
print("📋 各観測値（すべての測定値）:")

# 表形式で表示
df = pd.DataFrame({
    '番号': range(1, n+1),
    '観測値 (cm)': np.round(sample, 1)
})
# 5列に整形して表示
cols = 5
rows_per_col = n // cols
for i in range(rows_per_col):
    row_str = ""
    for j in range(cols):
        idx = i + j * rows_per_col
        if idx < n:
            row_str += f"  [{idx+1:2d}] {sample[idx]:.1f}cm"
    print(row_str)

In [ ]:
# 母集団と標本の比較可視化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：母集団
axes[0].hist(population, bins=50, color='#4A90D9', alpha=0.6, edgecolor='white')
axes[0].axvline(population.mean(), color='#E74C3C', linewidth=2.5, linestyle='--',
                label=f'母平均 μ = {population.mean():.1f}cm')
axes[0].set_title('【母集団】 N = 10,000人', fontsize=13, fontweight='bold')
axes[0].set_xlabel('身長 (cm)', fontsize=11)
axes[0].set_ylabel('人数', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# 右：標本
axes[1].hist(sample, bins=15, color='#E67E22', alpha=0.7, edgecolor='white')
axes[1].axvline(sample.mean(), color='#E74C3C', linewidth=2.5, linestyle='--',
                label=f'標本平均 x̄ = {sample.mean():.1f}cm')
axes[1].axvline(mu, color='#4A90D9', linewidth=2, linestyle=':', alpha=0.8,
                label=f'真の母平均 μ = {mu}cm')

# 各観測値をドットで表示
for obs in sample:
    axes[1].axvline(obs, color='#27AE60', alpha=0.15, linewidth=1)

axes[1].set_title(f'【標本】 n = {n}人 (無作為抽出)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('身長 (cm)', fontsize=11)
axes[1].set_ylabel('人数', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('母集団 vs 標本の比較', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔍 Step 3: 観測値を詳しく見る

標本中の**各測定値が「観測値」**です。観測値から統計量（平均・標準偏差など）を計算し、母集団のパラメータを推定します。

In [ ]:
# 観測値の詳細分析
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ① ドットプロット（各観測値を1点として表示）
ax = axes[0]
y_jitter = np.random.uniform(-0.3, 0.3, n)
scatter = ax.scatter(sample, y_jitter, c=sample, cmap='RdYlBu_r',
                     s=60, alpha=0.8, edgecolors='gray', linewidth=0.5)
ax.axvline(sample.mean(), color='red', linewidth=2, linestyle='--',
           label=f'x̄ = {sample.mean():.1f}cm')
ax.set_xlabel('身長 (cm)', fontsize=11)
ax.set_title('各観測値のドットプロット', fontsize=12, fontweight='bold')
ax.set_yticks([])
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.colorbar(scatter, ax=ax, label='身長 (cm)')

# ② 箱ひげ図
ax = axes[1]
bp = ax.boxplot(sample, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#E67E22', alpha=0.6),
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(linewidth=1.5),
                flierprops=dict(marker='o', color='red', alpha=0.7))
ax.axhline(mu, color='#4A90D9', linewidth=2, linestyle=':', label=f'母平均μ={mu}cm')
ax.set_title('箱ひげ図（標本）', fontsize=12, fontweight='bold')
ax.set_ylabel('身長 (cm)', fontsize=11)
ax.set_xticks([])
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

# 統計量の注釈
stats_text = (f"最小値: {sample.min():.1f}cm\n"
              f"Q1:    {np.percentile(sample,25):.1f}cm\n"
              f"中央値: {np.median(sample):.1f}cm\n"
              f"Q3:    {np.percentile(sample,75):.1f}cm\n"
              f"最大値: {sample.max():.1f}cm")
ax.text(1.3, np.median(sample), stats_text, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# ③ 母数 vs 標本統計量の比較棒グラフ
ax = axes[2]
categories = ['平均値 (cm)', '標準偏差 (cm)']
true_vals = [mu, sigma]
sample_vals = [sample.mean(), sample.std(ddof=1)]

x = np.arange(len(categories))
width = 0.35
bars1 = ax.bar(x - width/2, true_vals, width, label='母数（真の値）',
               color='#4A90D9', alpha=0.8)
bars2 = ax.bar(x + width/2, sample_vals, width, label='標本統計量（推定値）',
               color='#E67E22', alpha=0.8)

for bar, val in zip(bars1, true_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
for bar, val in zip(bars2, sample_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')

ax.set_title('母数 vs 標本統計量', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=10)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(true_vals + sample_vals) * 1.25)

plt.suptitle('観測値の分析', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 📐 Step 4: 標本サイズの影響を理解する

標本サイズ `n` が大きいほど、標本平均は母平均に近づきます（**大数の法則**）。

複数の標本サイズで繰り返し抽出してみましょう。

In [ ]:
# 標本サイズの違いによる推定精度の比較
sample_sizes = [5, 10, 30, 50, 100, 500]
n_trials = 1000  # 各サイズで1000回抽出

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

results = {}

for i, n_size in enumerate(sample_sizes):
    # n_trials回繰り返し標本平均を計算
    sample_means = [np.random.choice(population, n_size, replace=False).mean()
                    for _ in range(n_trials)]
    results[n_size] = sample_means

    ax = axes[i]
    ax.hist(sample_means, bins=40, color=plt.cm.viridis(i/len(sample_sizes)),
            alpha=0.75, edgecolor='white', linewidth=0.3)
    ax.axvline(mu, color='red', linewidth=2, linestyle='--', label=f'μ = {mu}')

    se = np.std(sample_means)
    ax.set_title(f'n = {n_size}\n標準誤差 SE = {se:.2f}cm', fontsize=11, fontweight='bold')
    ax.set_xlabel('標本平均 (cm)', fontsize=9)
    ax.set_ylabel('頻度', fontsize=9)
    ax.set_xlim(155, 187)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'標本サイズと標本平均の分布（各{n_trials}回抽出）\n nが大きいほど分布が狭くなる＝推定精度が高くなる',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 標準誤差と標本サイズの関係
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ses = [np.std(results[n]) for n in sample_sizes]
theoretical_ses = [sigma / np.sqrt(n) for n in sample_sizes]

# 左：標準誤差 vs 標本サイズ
axes[0].plot(sample_sizes, ses, 'o-', color='#E67E22', linewidth=2,
             markersize=8, label='実測の標準誤差')
axes[0].plot(sample_sizes, theoretical_ses, 's--', color='#4A90D9', linewidth=2,
             markersize=8, label=f'理論値 σ/√n (σ={sigma})')
axes[0].set_xlabel('標本サイズ n', fontsize=12)
axes[0].set_ylabel('標準誤差 SE (cm)', fontsize=12)
axes[0].set_title('標本サイズと標準誤差の関係', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

for n, se in zip(sample_sizes, ses):
    axes[0].annotate(f'SE={se:.2f}', (n, se), textcoords='offset points',
                     xytext=(5, 8), fontsize=8, color='#E67E22')

# 右：概念図（母集団→標本→観測値の階層）
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# 母集団円
pop_circle = plt.Circle((5, 7.5), 2.2, fill=True,
                         facecolor='#AED6F1', edgecolor='#2E86C1', linewidth=2.5)
ax.add_patch(pop_circle)
ax.text(5, 8.5, '母集団', ha='center', va='center', fontsize=13, fontweight='bold', color='#1A5276')
ax.text(5, 7.5, f'N={N:,}人\nμ={mu}cm, σ={sigma}cm', ha='center', va='center', fontsize=9, color='#1A5276')

# 矢印（抽出）
ax.annotate('', xy=(5, 5.2), xytext=(5, 5.7),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2.5))
ax.text(5.3, 5.45, '無作為抽出', fontsize=9, color='#E74C3C', fontweight='bold')

# 標本円
smp_circle = plt.Circle((5, 4), 1.4, fill=True,
                         facecolor='#FAD7A0', edgecolor='#E67E22', linewidth=2.5)
ax.add_patch(smp_circle)
ax.text(5, 4.5, '標本', ha='center', va='center', fontsize=12, fontweight='bold', color='#7D6608')
ax.text(5, 3.7, f'n={n}人', ha='center', va='center', fontsize=10, color='#7D6608')

# 矢印（観測）
ax.annotate('', xy=(5, 2.1), xytext=(5, 2.7),
            arrowprops=dict(arrowstyle='->', color='#27AE60', lw=2.5))
ax.text(5.3, 2.4, '各個体を測定', fontsize=9, color='#27AE60', fontweight='bold')

# 観測値ボックス
obs_rect = mpatches.FancyBboxPatch((2.5, 0.5), 5, 1.5,
                                    boxstyle='round,pad=0.1',
                                    facecolor='#D5F5E3', edgecolor='#1E8449', linewidth=2)
ax.add_patch(obs_rect)
ax.text(5, 1.5, '観測値', ha='center', va='center', fontsize=12, fontweight='bold', color='#1E8449')
ax.text(5, 0.9, '175.2, 168.7, 182.1, ...', ha='center', va='center', fontsize=9, color='#1E8449')

ax.set_title('概念の関係図', fontsize=13, fontweight='bold')

plt.suptitle('標準誤差と母集団・標本・観測値の関係', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎲 Step 5: インタラクティブ体験 - 自分で標本を抽出してみよう

標本サイズを変えて、標本平均がどのくらい母平均に近くなるか確認しましょう。

In [ ]:
# ===== 自分で試してみよう！=====
# 👇 ここの数値を変えて実行してみてください

MY_SAMPLE_SIZE = 30   # ← 標本サイズを変えてみよう（5〜5000の間）
MY_SEED = 99          # ← 乱数シードを変えると別の標本が得られる

# =================================
np.random.seed(MY_SEED)
my_sample = np.random.choice(population, MY_SAMPLE_SIZE, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 標本ヒストグラム
axes[0].hist(my_sample, bins=max(5, MY_SAMPLE_SIZE//5), color='#9B59B6',
             alpha=0.7, edgecolor='white')
axes[0].axvline(my_sample.mean(), color='red', linewidth=2.5, linestyle='--',
                label=f'標本平均 x̄ = {my_sample.mean():.2f}cm')
axes[0].axvline(mu, color='blue', linewidth=2, linestyle=':', alpha=0.8,
                label=f'真の母平均 μ = {mu}cm')
axes[0].set_title(f'あなたの標本 (n={MY_SAMPLE_SIZE})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('身長 (cm)', fontsize=11)
axes[0].set_ylabel('人数', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# 結果サマリー
ax = axes[1]
ax.axis('off')

error = abs(my_sample.mean() - mu)
error_pct = (error / mu) * 100
se_theoretical = sigma / np.sqrt(MY_SAMPLE_SIZE)

summary = [
    ('母集団の真の平均 (μ)', f'{mu:.2f} cm'),
    ('あなたの標本平均 (x̄)', f'{my_sample.mean():.2f} cm'),
    ('推定誤差 |x̄ - μ|', f'{error:.2f} cm ({error_pct:.1f}%)'),
    ('理論的な標準誤差 σ/√n', f'{se_theoretical:.2f} cm'),
    ('標本の標準偏差 (s)', f'{my_sample.std(ddof=1):.2f} cm'),
    ('標本サイズ (n)', f'{MY_SAMPLE_SIZE} 人'),
]

ax.text(0.5, 0.95, ' 結果サマリー', transform=ax.transAxes,
        ha='center', va='top', fontsize=14, fontweight='bold')

for j, (label, value) in enumerate(summary):
    y_pos = 0.80 - j * 0.12
    color = '#E74C3C' if '誤差' in label else '#2C3E50'
    ax.text(0.05, y_pos, label, transform=ax.transAxes,
            ha='left', va='center', fontsize=11, color='#555')
    ax.text(0.95, y_pos, value, transform=ax.transAxes,
            ha='right', va='center', fontsize=12, fontweight='bold', color=color)
    ax.plot([0.03, 0.97], [y_pos - 0.05, y_pos - 0.05],
            transform=ax.transAxes, color="#ddd", linewidth=0.8)

# 評価コメント
if error <= se_theoretical:
    comment = " 優秀！標準誤差以内に収まっています"
    c_color = '#27AE60'
elif error <= 2 * se_theoretical:
    comment = " 良好！2×標準誤差以内に収まっています"
    c_color = '#E67E22'
else:
    comment = "️ 今回は誤差が大きめです（運が悪かった）"
    c_color = '#E74C3C'

ax.text(0.5, 0.05, comment, transform=ax.transAxes,
        ha='center', va='bottom', fontsize=11, color=c_color, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor=c_color))

plt.suptitle(f'あなたの標本抽出結果 (seed={MY_SEED})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n💡 ヒント: MY_SAMPLE_SIZE を大きくすると推定誤差が小さくなります！")
print(f"　　MY_SEED を変えると別の標本が抽出されます。")


## 📝 まとめ

このノートブックで学んだことを整理しましょう。

In [ ]:
# まとめの表示
print("="*55)
print("  📚 統計学の基礎概念まとめ")
print("="*55)
print()
print("┌─────────────────────────────────────────────────┐")
print("│  概念        記号   意味                        │")
print("├─────────────────────────────────────────────────┤")
print("│  母集団      N      調査対象の全体集合           │")
print("│  母平均      μ      母集団全体の平均値           │")
print("│  母標準偏差  σ      母集団全体のばらつき         │")
print("├─────────────────────────────────────────────────┤")
print("│  標本        n      母集団から抽出した部分集合   │")
print("│  標本平均    x̄      標本の平均値（μの推定値）   │")
print("│  標本標準偏差 s     標本のばらつき（σの推定値） │")
print("├─────────────────────────────────────────────────┤")
print("│  観測値      xᵢ     各個体の測定値              │")
print("│  標準誤差    SE     x̄のばらつき = σ/√n        │")
print("└─────────────────────────────────────────────────┘")
print()
print("🔑 重要なポイント:")
print("  1. 標本サイズ n が大きいほど、推定精度が上がる")
print("  2. 無作為抽出により、偏りのない推定ができる")
print("  3. 標準誤差 SE = σ/√n（nの平方根に反比例）")
print("  4. 観測値の集まりが標本、全体が母集団")